### 1. Load the raw CSV

Keep every source value as text, including empty cells and leading zeros. The raw file and `raw_df` stay unchanged.

In [20]:
import pandas as pd

raw_df = pd.read_csv(
    "output/invoice-1791938597297.table.csv",
    header=None,
    dtype="string",
    keep_default_na=False,
)

### 2. Define the table structure

Remove the invoice header row and unwanted column, then apply the agreed column names. Keep source row indices so rows can be traced back to `raw_df`.

In [21]:
clean_df = (
    raw_df.drop(index=0, columns=[6])
    .rename(columns={
        0: "Item Id",
        1: "MLCC ID",
        2: "Size/Pack",
        3: "Item",
        4: "Shelf Price",
        5: "Base Price",
        7: "Shipped Cases",
        8: "Shipped Bottles",
        9: "Cost",
        10: "Total Price",
    })
    .copy()
)

### 3. Keep rows with valid identifiers

Both IDs must contain only ASCII digits. Keep IDs as strings to preserve leading zeros. Empty values in other columns do not cause row removal.

In [22]:
valid_ids = (
    clean_df["Item Id"].str.fullmatch(r"[0-9]+", na=False)
    & clean_df["MLCC ID"].str.fullmatch(r"[0-9]+", na=False)
)
clean_df = clean_df.loc[valid_ids].copy()

### 4. Clean text

Remove newlines from Size/Pack and replace them with spaces in Item. No further description or pack normalization is applied yet.

In [23]:
clean_df["Size/Pack"] = clean_df["Size/Pack"].str.replace("\n", "", regex=False)
clean_df["Item"] = clean_df["Item"].str.replace("\n", " ", regex=False)

### 5. Clean and convert shipped quantities

Remove only the known Azure selection markers and surrounding whitespace. Empty quantities become null. Every nonempty value must contain only ASCII digits; decimals, negative values, unknown text, and internal separators raise an error showing the source rows. Convert valid quantities to nullable integers.

In [24]:
for column in ["Shipped Cases", "Shipped Bottles"]:
    quantities = (
        clean_df[column].astype("string")
        .str.replace(":selected:", "", regex=False)
        .str.replace(":unselected:", "", regex=False)
        .str.strip()
        .replace("", pd.NA)
    )
    invalid = quantities.notna() & ~quantities.str.fullmatch(r"[0-9]+", na=False)
    if invalid.any():
        raise ValueError(
            f"Invalid quantities in {column} (source row: value): {quantities[invalid].to_dict()}"
        )
    clean_df[column] = pd.to_numeric(quantities, errors="raise").astype("Int64")

### 6. Clean and convert prices

For Total Price, extract the first valid dollar amount from the raw cell. This preserves `$32.40` from `Stock $32.40`. Cells containing only stock text or other nonprice text become null. Normalize case and whitespace before extraction.

Convert all price columns to nullable floats. Other price columns still raise on unexpected nonnumeric text. No amounts are calculated.

In [25]:
total_price = (
    raw_df.loc[clean_df.index, 10]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
clean_df["Total Price"] = total_price.str.extract(
    r"(\$(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]+)\.[0-9]{2})(?![0-9.,])",
    expand=False,
)

for column in ["Shelf Price", "Base Price", "Cost", "Total Price"]:
    prices = (
        clean_df[column].astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("", pd.NA)
    )
    clean_df[column] = pd.to_numeric(prices, errors="raise").astype("Float64")

### 7. Review the result

Run the notebook from top to bottom after changing the source CSV or any cleaning rule. This notebook does not write over the source CSV or upload rows to a database.

In [26]:
clean_df

,Item Id,MLCC ID,Size/Pack,Item,Shelf Price,Base Price,Shipped Cases,Shipped Bottles,Cost,Total Price
1,61094,1645,50 ml/60,1800 Blanco Tequila,2.49,2.22,1,0,2.11,126.6
2,61089,1644,50 ml/60,1800 Reposado Tequila,2.79,2.49,0,0,2.37,<NA>
3,56700,14303,375 ml/24,Canadian Rich & Rare,3.99,3.57,1,0,3.38,81.12
4,56522,14307,750 ml/12,Canadian Rich & Rare,7.49,6.68,0,0,6.35,<NA>
5,35455,7829,100 ml/48,Fireball Cinnamon PI,1.99,1.78,1,0,1.69,81.12
6,35547,16254,10pk/50ml/12,Fireball Cinnamon PL,10.0,8.92,1,0,8.48,101.76
7,35004,1438,200 ml/48,Fireball Cinnamon PI,3.99,3.57,1,0,3.38,162.24
8,35054,1439,375 ml/24,Fireball Cinnamon PI,6.99,6.24,0,0,5.93,<NA>
9,42490,101090,750 ml/12,Fireball Torch'd Vanilla,12.99,11.61,0,0,11.02,<NA>
10,42609,101325,50 ml/12,Fireball Torch'd Vanilla 10Pk,10.0,8.92,0,0,8.48,<NA>
